# Building K Detection — My Movie 2_1
Runs V2 (red mask) and V3 (blue mask) on `My movie 2_1.mp4` and saves both output videos to Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

V2_MODEL   = "/content/drive/MyDrive/COMP4010_finalprj/building_K/models/building_k_v2_best.pt"
V3_MODEL   = "/content/drive/MyDrive/COMP4010_finalprj/building_K/models/building_k_v3_best.pt"
VIDEO      = "/content/drive/MyDrive/COMP4010_finalprj/building_K/building_k_raw/My movie 2_1.mp4"
OUTPUT_DIR = "/content/drive/MyDrive/COMP4010_finalprj/building_K/video_predictions"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"V2 model : {'OK' if os.path.exists(V2_MODEL) else 'NOT FOUND'}")
print(f"V3 model : {'OK' if os.path.exists(V3_MODEL) else 'NOT FOUND'}")
print(f"Video    : { 'OK' if os.path.exists(VIDEO)    else 'NOT FOUND'}")
print(f"Output   : {OUTPUT_DIR}")

V2 model : OK
V3 model : OK
Video    : OK
Output   : /content/drive/MyDrive/COMP4010_finalprj/building_K/video_predictions


In [7]:
import cv2
import numpy as np
from ultralytics import YOLO

model_v2 = YOLO(V2_MODEL)
model_v3 = YOLO(V3_MODEL)

configs = [
    (model_v2, "Model V2", (0, 0, 255), "my_movie_v2.mp4"),
    (model_v3, "Model V3", (255, 0, 0), "my_movie_v3.mp4"),
]

cap    = cv2.VideoCapture(VIDEO)
fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print(f"Video: {total} frames at {fps:.0f}fps ({width}x{height})\n")

for model, label, color, out_name in configs:
    out_path = os.path.join(OUTPUT_DIR, out_name)
    writer   = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    print(f"Running {label}...", end=" ", flush=True)

    for fi, result in enumerate(model(VIDEO, stream=True, conf=0.90, verbose=False), 1):
        frame = result.orig_img.copy()

        if result.masks is not None:
            for mask in result.masks.data:
                mask_np      = mask.cpu().numpy()
                mask_resized = cv2.resize(mask_np, (width, height))
                mask_bool    = mask_resized > 0.5
                overlay      = frame.copy()
                overlay[mask_bool] = color
                frame = cv2.addWeighted(frame, 0.55, overlay, 0.45, 0)
                contours, _ = cv2.findContours(
                    (mask_resized * 255).astype(np.uint8),
                    cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(frame, contours, -1, color, 3)

        if result.boxes is not None and len(result.boxes) > 0:
            conf = result.boxes.conf[0].item()
            cv2.putText(frame, f"building_k  {conf:.2f}", (20, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 2, cv2.LINE_AA)

        cv2.putText(frame, label, (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 2, cv2.LINE_AA)

        writer.write(frame)
        if fi % 200 == 0:
            print(f"{fi/total*100:.0f}%...", end=" ", flush=True)

    writer.release()
    print(f"done ✓  →  {out_name}")

print(f"\nSaved to: {OUTPUT_DIR}")

Video: 1218 frames at 60fps (3840x2160)

Running Model V2... 16%... 33%... 49%... 66%... 82%... 99%... done ✓  →  my_movie_v2.mp4
Running Model V3... 16%... 33%... 49%... 66%... 82%... 99%... done ✓  →  my_movie_v3.mp4

Saved to: /content/drive/MyDrive/COMP4010_finalprj/building_K/video_predictions
